# Feature Engineering: ColumnTransformer and Pipeline

### Do You Want to Build a Snowman?

Prepare the inputs for a model that predicts snowman height.

<img src="../assets/olaf.jpeg" width="300" alt="Snowman illustration">

This deliberately tiny, fictional dataset is a **preprocessing exercise**, not evidence that lunch or dinner predicts snowman height. Treat all rows as a supplied training sample; do not report a model generalization score from these 14 examples.

**Business connection:** in a housing dataset, the same steps could impute missing floor areas, scale numerical measurements, and encode property types before predicting price. The workflow transfers; the useful features and preprocessing choices depend on the task.

#### Load Packages

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

#### Load Data

In [ ]:
data = {
    "temp": [-3, 5, 0, 7, 3, -1, 1, None, -6, 3, 0, -1, None, -2],
    "lunch": [
        "soup",
        "sandwich",
        "soup",
        "burger",
        "sandwich",
        "soup",
        "cereal",
        "salad",
        "sandwich",
        "burger",
        "soup",
        "cereal",
        "burger",
        "soup",
    ],
    "dinner": [
        "pizza",
        "pizza",
        "noodles",
        None,
        "fishsticks",
        "pizza",
        None,
        "fishsticks",
        "noodles",
        "pizza",
        None,
        "pizza",
        "fishsticks",
        "pizza",
    ],
    "precipitation": [
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "no",
    ],
    "height_snowman_cm": [100, 0, 75, 0, 20, 25, 0, 35, 170, 0, 85, 85, 45, 0],
}

df_train = pd.DataFrame(data=data).fillna(np.nan)
df_train

## One Possible Solution

1. Separate `df_train` into `X_train` and `y_train`; the target is `height_snowman_cm`.
2. Identify numeric, nominal, and binary inputs, then inspect missing values.
3. Impute `temp` with its training median and `dinner` with its most frequent value. Row order has no time meaning here, so do not interpolate between unrelated observations.
4. Scale `temp` and one-hot encode `lunch`, `dinner`, and `precipitation`.
5. Use a `ColumnTransformer` with an impute-then-scale numeric pipeline and an impute-then-encode categorical pipeline. Request pandas output.
6. Produce `X_train_fe` with numerical values, no missing values, the same row count and index as `X_train`, and no target column.

These are example choices, not the only valid answer: explain why your transformations suit each variable. Fit on training data only. To transform new rows later, reuse the fitted objects instead of recomputing statistics. Make sure each input feature is selected exactly once.

1. **Separate the DataFrame `df_train` into `X_train` and `y_train`**

In [ ]:
target = "height_snowman_cm"

In [ ]:
# Feature matrix
X_train = df_train.drop(target, axis=1)
y_train = df_train[target]

2. **Preprocess `X_train`**

In [ ]:
# Fill missing temperatures before scaling them.
numeric_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

Apply imputation before encoding for all categorical inputs. A shared pipeline works because they use the same strategy. Keep the numeric pipeline in a separate branch.

The encoder keeps all category indicators here. With `handle_unknown="ignore"`, an unseen category becomes all zeros for that input column; monitor such cases when using the pipeline on new data.

In [ ]:
# Define the steps in their execution order: impute, then encode.
categorical_steps = [
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(sparse_output=False, handle_unknown="ignore")),
]

In [ ]:
# Combine the categorical steps into a reusable pipeline.
categorical_pipeline = Pipeline(steps=categorical_steps)
categorical_pipeline

In [ ]:
# Route each input once, including precipitation, to its pipeline.
transformers = [
    ("temperature", numeric_pipeline, ["temp"]),
    ("categories", categorical_pipeline, ["lunch", "dinner", "precipitation"]),
]

In [ ]:
fe_column_transformer = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
    verbose_feature_names_out="{transformer_name}_{feature_name}",
).set_output(transform="pandas")
fe_column_transformer

In [ ]:
fe_column_transformer.fit(X_train)

In [ ]:
X_train_fe = fe_column_transformer.transform(X_train)
assert X_train_fe.index.equals(y_train.index)
assert not X_train_fe.isna().any().any()
X_train_fe

### Why This Pipeline?

In this pipeline, we impute before scaling or encoding so the later steps receive complete inputs. This is a deliberate sequence, not a universal requirement: some scalers preserve missing values, and encoders can treat them as a category. `ColumnTransformer` then joins the numeric and categorical outputs. Median temperature imputation and mode category imputation are simple baselines, not uniquely correct choices.

### Reuse on New Inputs

Unlike the introductory solution, this encoder retains every category indicator and explicitly accepts unknown categories. `wrap` becomes an all-zero **lunch block**, not an all-zero row. This keeps the schema stable but does not tell the model what the new category means.

In [ ]:
# New inputs only: the fitted transformers must not learn from these rows.
X_new = pd.DataFrame(
    {
        "temp": [np.nan, 2.0],
        "lunch": ["soup", "wrap"],
        "dinner": [np.nan, "pizza"],
        "precipitation": ["yes", "no"],
    },
    index=["known", "new_category"],
)
X_new

In [ ]:
X_new_fe = fe_column_transformer.transform(X_new)
assert X_new_fe.columns.equals(X_train_fe.columns)
assert X_new_fe.index.equals(X_new.index)
assert not X_new_fe.isna().any().any()
# Only the indicators for the unseen lunch should all be zero.
lunch_columns = X_new_fe.columns.str.startswith("categories_lunch_")
assert (X_new_fe.loc["new_category", lunch_columns] == 0).all()
X_new_fe